In [ ]:
# ==== 0) Setup (T4-friendly) ====
!pip -q install --upgrade pip
!pip -q install "torch==2.4.0" "torchvision==0.19.0" --index-url https://download.pytorch.org/whl/cu121
!pip -q install "transformers>=4.45.0" "accelerate>=0.34.0" "safetensors>=0.4.4" \
               "bitsandbytes>=0.43.1" "einops>=0.7.0" "open-clip-torch>=2.24.0" pillow    

In [ ]:
import torch, os
from transformers import AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig
from PIL import Image, ImageOps
import matplotlib.pyplot as plt

import os
import csv

In [ ]:
# Check GPU
assert torch.cuda.is_available(), "Please enable a GPU runtime in Colab."
print("CUDA device:", torch.cuda.get_device_name(0))
torch.backends.cuda.matmul.allow_tf32 = True

# ==== 2) Model choice (HF-converted weights for smooth loading) ====
MODEL_ID = "chaoyinshe/llava-med-v1.5-mistral-7b-hf"  # public HF conversion of microsoft/llava-med-v1.5-mistral-7b

# 4-bit quantization config (T4-safe)
bnb_4bit = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,  # T4 doesn't support bfloat16 well; fp16 is safer
)

In [ ]:
# ==== 3) Load model & processor ====
print("Loading model… (this can take a bit)")
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_4bit,
    device_map="auto",
    low_cpu_mem_usage=True,
    attn_implementation=None,  # no flash-attn on T4
)

processor = AutoProcessor.from_pretrained(MODEL_ID)
print("Loaded.")

In [ ]:
# ==== 4) Helper: build chat prompt (works across tokenizer/proc variants) ====
def build_prompt(processor, question: str):
    messages = [
        {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": question}]}
    ]
    # Newer Transformers: processor.apply_chat_template; fallback to tokenizer if needed
    if hasattr(processor, "apply_chat_template"):
        return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    elif hasattr(processor, "tokenizer") and hasattr(processor.tokenizer, "apply_chat_template"):
        return processor.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        # Minimal fallback prompt
        return f"[INST] <image>\n{question} [/INST]"

In [ ]:
import re

def extract_location(text):
    # remove everything before first occurrence of "located"
    text = re.sub(r".*located (on|in)\s+", "", text, flags=re.IGNORECASE)

    # remove filler phrases
    text = text.replace("the patient's ", "")
    text = text.replace("patient's ", "")
    text = text.replace("the ", "")

    # final clean
    return text.strip().lower().rstrip('.')

In [ ]:
# --- CONFIG ---
image_folder = "./data/LLaVA-Med/images/test"
output_file = "./data/LLaVA-Med/Results_LLaVA_051225_1.csv"

#question = """You are a clinical diagnostic assistant. Analyze the provided image for visual evidence of an Adverse Drug Effect (ADE).
#If an ADE is detected, provide the location of the ADE. """

question = "Which side of the leg is shown in the image?"

print(f"Prompt: {question}")

# --- FUNCTION TO GENERATE CAPTION FOR ONE IMAGE ---
def generate_caption(image_path, model, processor, question):
    image = Image.open(image_path).convert("RGB")

    #image = ImageOps.mirror(image)

    plt.imshow(image)
    plt.axis("off")   # removes axes
    plt.show()
    
    # Build prompt & inputs
    prompt = build_prompt(processor, question)
    fresh = processor(images=[image], text=prompt, return_tensors="pt")

    inputs = {}

    # Move tensors correctly
    if "input_ids" in fresh:
        inputs["input_ids"] = fresh["input_ids"].to(model.device, dtype=torch.long)
    if "attention_mask" in fresh:
        inputs["attention_mask"] = fresh["attention_mask"].to(model.device)
    if "pixel_values" in fresh:
        inputs["pixel_values"] = fresh["pixel_values"].to(model.device, dtype=torch.float16)
    if "image_sizes" in fresh:
        inputs["image_sizes"] = fresh["image_sizes"].to(model.device)
    for k in ("token_type_ids", "position_ids"):
        if k in fresh:
            inputs[k] = fresh[k].to(model.device, dtype=torch.long)

    # Pad/EOS ids
    pad_id = getattr(getattr(processor, "tokenizer", None), "pad_token_id", None)
    eos_id = getattr(getattr(processor, "tokenizer", None), "eos_token_id", None)
    if pad_id is None and eos_id is not None:
        pad_id = eos_id

    # Generate caption
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            temperature=None,
            pad_token_id=pad_id,
            eos_token_id=eos_id,
        )

    decode_fn = getattr(processor, "decode", None) or processor.tokenizer.decode
    text = decode_fn(output_ids[0], skip_special_tokens=True)

    return text.strip()


# --- MAIN LOOP ---
captions = []
image_files = [f for f in os.listdir(image_folder) if f.lower().endswith((".png", ".jpg", ".jpeg", ".webp"))]
image_files.sort()  # sort alphabetically for consistency

for img_name in image_files:
    img_path = os.path.join(image_folder, img_name)
    try:
        caption = generate_caption(img_path, model, processor, question)

        # Split at the marker [/INST]
        if '[/INST]' in caption:
            before, after = caption.split('[/INST]', 1)  # split only once
            caption = after.strip()     # clean up leading/trailing spaces
        else:
            caption = caption

        caption = extract_location(caption)

        captions.append({"img_name": img_name, "caption": caption})
        print(f"✅ {img_name}: {caption}")
    except Exception as e:
        print(f"⚠️ Error processing {img_name}: {e}")

# --- WRITE OUTPUT AS CSV ---
with open(output_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["img_name", "caption"])
    writer.writeheader()
    writer.writerows(captions)

print(f"\n--- Captions written to {output_file} ---")